# 02 — `__init_subclass__` et décorateurs de classe

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `__init_subclass__` pour intervenir à la création d'une sous-classe ;
- écrire des décorateurs de classe qui transforment ou enregistrent une classe ;
- implémenter un **registry** automatique (plugin system) ;
- comparer ces alternatives aux métaclasses et savoir **pourquoi** les préférer ;
- combiner `__init_subclass__` avec les descripteurs pour un mini-framework.


## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les type hints modernes (`int | None`, génériques, `Protocol`, `TypeVar`) ;
- les dataclasses (`@dataclass`, `field`, `frozen=True`, `slots=True`) ;
- les décorateurs de fonction et de classe, et les gestionnaires de contexte ;
- les tests avec `pytest` (fixtures, paramétrage, monkeypatch) ;
- le packaging avec `pyproject.toml` et `uv` ;
- le logging (module `logging`, handlers, formatters) ;
- les bases de SQL et `sqlite3`, les expressions régulières.
- les descripteurs (notebook précédent).

Notions que nous allons **introduire ou approfondir** ici :

- `__init_subclass__` et son utilité concrète ;
- les décorateurs de classe avancés ;
- le pattern **registry** / plugin system.


## Plan

1. `__init_subclass__` : 95 % des cas d'usage de métaclasse
2. Décorateurs de classe : rappel et cas avancés
3. Pattern 1 : registry automatique
4. Pattern 2 : validation de classe
5. Pattern 3 : injection d'attributs
6. Pattern 4 : décorateur de méthodes en masse
7. Comparaison : property, dataclass, `__init_subclass__`, métaclasse
8. Synthèse
9. Exercices


---

## 1. `__init_subclass__` — un hook automatique de création de sous-classe

La **PEP 487** (Python 3.6) introduit `__init_subclass__`, une méthode de classe **automatiquement appelée** à chaque fois qu'une **sous-classe** est définie. C'est le moyen le plus simple d'intervenir à la création d'une classe sans écrire de métaclasse.

La signature est : `def __init_subclass__(cls, **kwargs) -> None`. Elle est implicitement `classmethod` (vous n'avez pas besoin du décorateur).

In [ ]:
class Base:
    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        print(f"Sous-classe créée : {cls.__name__}")

class A(Base): pass
class B(A): pass

Ce qu'il faut noter :

- `cls` est la **sous-classe** en cours de création (pas `Base`).
- `__init_subclass__` est appelée **avant** que la classe ne soit rendue accessible par son nom.
- Il faut toujours appeler `super().__init_subclass__(**kwargs)` pour la compatibilité avec les   chaînes d'héritage multiples.

### Paramètres d'héritage personnalisés

On peut passer des arguments **au moment de l'héritage** (sur la ligne `class ... :`) qui seront transmis à `__init_subclass__`.

In [ ]:
class Plugin:
    def __init_subclass__(cls, *, nom: str, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        cls.nom_plugin = nom
        print(f"Plugin enregistré : {nom}")

class ExportCSV(Plugin, nom="csv"):
    pass

class ExportJSON(Plugin, nom="json"):
    pass

print(ExportCSV.nom_plugin)
print(ExportJSON.nom_plugin)

---

## 2. Décorateurs de classe — rappel et cas avancés

Un décorateur de classe est une fonction qui prend une classe et retourne (potentiellement) une classe modifiée. Vous en connaissez déjà les plus célèbres : `@dataclass`, `@total_ordering`, `@runtime_checkable`.

Contrairement à `__init_subclass__`, un décorateur de classe **s'applique uniquement à la classe décorée** (pas aux sous-classes), sauf si on le propage explicitement.

In [ ]:
def log_creation(cls):
    print(f"Classe créée : {cls.__name__}")
    return cls

@log_creation
class Service:
    pass

### Décorateur qui modifie la classe

Un décorateur peut ajouter des méthodes, des attributs, ou envelopper des méthodes existantes.

In [ ]:
def ajoute_repr(cls):
    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

@ajoute_repr
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

print(Point(1.0, 2.0))

---

## 3. Pattern — Registry automatique (plugin system)

Un **registry** est un dictionnaire qui enregistre toutes les sous-classes d'une classe de base. C'est le cœur de presque tous les systèmes de plugin Python. On peut l'implémenter avec `__init_subclass__` en 5 lignes :

In [ ]:
class Exporteur:
    _registry: dict[str, type["Exporteur"]] = {}

    def __init_subclass__(cls, *, format: str, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        Exporteur._registry[format] = cls

    def export(self, data) -> str:
        raise NotImplementedError

class ExportJSON(Exporteur, format="json"):
    def export(self, data) -> str:
        import json
        return json.dumps(data)

class ExportCSV(Exporteur, format="csv"):
    def export(self, data) -> str:
        return ";".join(str(v) for v in data)

print(Exporteur._registry)

# On peut maintenant instancier par nom
exporter = Exporteur._registry["json"]()
print(exporter.export({"x": 1}))

Ce pattern permet à de **nouveaux plugins** de s'enregistrer automatiquement juste en héritant de la classe de base, sans aucune ligne de code supplémentaire dans le code appelant. C'est exactement le même pattern que celui utilisé par `click`, `pytest` (plugins), `sphinx` (directives).

---

## 4. Pattern — Validation de classe à la déclaration

On peut utiliser `__init_subclass__` pour **imposer un contrat** à toutes les sous-classes : vérifier qu'elles définissent bien certains attributs ou méthodes.

In [ ]:
class Handler:
    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        requis = {"name", "handle"}
        manquants = requis - set(vars(cls))
        if manquants:
            raise TypeError(f"{cls.__name__} doit définir {manquants}")

class EmailHandler(Handler):
    name = "email"
    def handle(self, msg): print(msg)

print("OK :", EmailHandler.name)

In [ ]:
# Une classe qui oublie un attribut est rejetée immédiatement
try:
    class MauvaisHandler(Handler):
        name = "bug"
        # oubli de handle
except TypeError as e:
    print("Refusé :", e)

---

## 5. Pattern — Injection automatique d'attributs

Combiner `__init_subclass__` avec `__set_name__` permet d'implémenter facilement un **système de schéma** qui découvre automatiquement les champs déclarés.

In [ ]:
class Field:
    def __set_name__(self, owner, name: str) -> None:
        self.name = name

class Schema:
    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        cls._fields = [
            v.name for v in vars(cls).values() if isinstance(v, Field)
        ]

class User(Schema):
    nom = Field()
    email = Field()
    age = Field()

print(User._fields)

---

## 6. Pattern — Décorateur appliqué en masse

On peut écrire un décorateur de classe qui **décore toutes les méthodes** publiques d'une classe. Très utile pour ajouter du tracing ou de l'audit à un service.

In [ ]:
import functools

def trace_toutes_methodes(cls):
    for nom, valeur in list(vars(cls).items()):
        if callable(valeur) and not nom.startswith("_"):
            @functools.wraps(valeur)
            def wrapper(self, *args, __f=valeur, __n=nom, **kwargs):
                print(f"> {cls.__name__}.{__n}({args}, {kwargs})")
                return __f(self, *args, **kwargs)
            setattr(cls, nom, wrapper)
    return cls

@trace_toutes_methodes
class Compteur:
    def __init__(self) -> None:
        self.n = 0

    def inc(self, v: int = 1) -> None:
        self.n += v

    def get(self) -> int:
        return self.n

c = Compteur()
c.inc(3)
print(c.get())

**Piège classique :** la closure `lambda`/`def` doit capturer la fonction originale par **paramètre par défaut** (`__f=valeur`) et non par capture de variable libre, sinon toutes les méthodes décorées appelleront la dernière de la boucle.

---

## 7. Comparaison des outils de métaprogrammation

| Outil | Complexité | Héritable ? | Cas d'usage typique |
|---|---|---|---|
| `@property` | très faible | oui | un attribut avec logique |
| **descripteur custom** | faible | oui | même logique sur plusieurs attributs |
| **décorateur de classe** | faible | non (sauf propagation) | modif ponctuelle d'une classe |
| **`@dataclass`** | faible | non par défaut | `__init__`, `__repr__`, `__eq__` auto |
| **`__init_subclass__`** | moyenne | oui (toutes les sous-classes) | registry, validation, injection |
| **métaclasse** | élevée | oui | quand rien d'autre ne suffit (ORMs, ABCs) |

**Règle d'or :** chaque ligne du tableau est une alternative à explorer **avant** la suivante. La métaclasse est un dernier recours — 95 % des problèmes se résolvent sans.

---

## 8. Synthèse

- `__init_subclass__` est **le** hook à connaître en 2026 : simple, explicite, héritable.
- Un décorateur de classe est parfait pour une **transformation ponctuelle** sans toucher à l'héritage.
- Les deux se combinent très bien avec les descripteurs et `@dataclass`.
- Avant d'écrire une métaclasse, demandez-vous si `__init_subclass__` ne suffit pas. Réponse :   oui dans 95 % des cas.


---

## 9. Exercices

### Exercice 1 — Registry de commandes CLI *(facile)*

Écrire une classe `Command` telle que chaque sous-classe s'enregistre dans un dictionnaire `Command.registry` sous la clé passée à la création (`class Foo(Command, nom="foo"):`). Ajouter une fonction `dispatch(nom, *args)` qui instancie la commande par nom et appelle sa méthode `run(*args)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Init_subclass_et_class_decorators", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class Command:
    registry: dict[str, type["Command"]] = {}

    def __init_subclass__(cls, *, nom: str, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        Command.registry[nom] = cls

    def run(self, *args) -> None:
        raise NotImplementedError

class Hello(Command, nom="hello"):
    def run(self, qui: str) -> None:
        print(f"bonjour {qui}")

def dispatch(nom: str, *args) -> None:
    Command.registry[nom]().run(*args)

dispatch("hello", "Ada")
```

</details>


### Exercice 2 — Décorateur `@auto_slots` *(moyen)*

Écrire un décorateur de classe `auto_slots` qui déduit `__slots__` de la signature du `__init__` (en lisant les annotations). Le décorateur doit créer une nouvelle classe avec `__slots__` et recopier tous les membres de l'ancienne.

**Indice :** regardez `inspect.get_annotations(cls.__init__)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Init_subclass_et_class_decorators", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import inspect

def auto_slots(cls):
    annotations = inspect.get_annotations(cls.__init__)
    # On retire 'return' si présent
    slots = tuple(k for k in annotations if k != "return")
    attrs = {k: v for k, v in vars(cls).items() if k not in ("__dict__", "__weakref__")}
    attrs["__slots__"] = slots
    return type(cls.__name__, cls.__bases__, attrs)

@auto_slots
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

p = Point(1.0, 2.0)
print(Point.__slots__)
try:
    p.z = 3
except AttributeError as e:
    print("slots actifs :", e)
```

</details>


### Exercice 3 — Validation de cohérence *(difficile)*

Écrire une classe `Serializable` dont chaque sous-classe doit :

1. Définir `to_dict` et `from_dict` (classmethod) ;
2. Avoir un attribut de classe `VERSION: int` ;
3. Lever `TypeError` à la création si une de ces conditions n'est pas respectée.

Tester qu'une classe oubliant un élément est bien refusée.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Init_subclass_et_class_decorators", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Serializable:
    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        # VERSION doit être présent dans la classe courante
        if "VERSION" not in vars(cls):
            raise TypeError(f"{cls.__name__} doit définir VERSION")
        if not isinstance(vars(cls)["VERSION"], int):
            raise TypeError("VERSION doit être un int")
        # to_dict et from_dict doivent être définis (éventuellement hérités du super)
        for nom in ("to_dict", "from_dict"):
            if not callable(getattr(cls, nom, None)):
                raise TypeError(f"{cls.__name__} doit définir {nom}")

class Article(Serializable):
    VERSION = 1

    def __init__(self, titre: str) -> None:
        self.titre = titre

    def to_dict(self) -> dict:
        return {"titre": self.titre}

    @classmethod
    def from_dict(cls, d: dict) -> "Article":
        return cls(d["titre"])

a = Article("Test")
print(a.to_dict())
```

</details>


### Exercice 4 — Mini-framework event bus *(deep dive)*

Écrire un système où toute méthode décorée par `@on("event_name")` est automatiquement enregistrée dans un dispatcher au moment de la création de la classe (via `__init_subclass__`). Implémenter `dispatcher.publish(event_name, *args)` qui appelle toutes les méthodes enregistrées.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Init_subclass_et_class_decorators", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
def on(event: str):
    def deco(fn):
        fn._event = event
        return fn
    return deco

class EventBus:
    _handlers: dict[str, list] = {}

    def __init_subclass__(cls, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        for nom, fn in vars(cls).items():
            event = getattr(fn, "_event", None)
            if event:
                EventBus._handlers.setdefault(event, []).append((cls, nom))

    @classmethod
    def publish(cls, event: str, *args) -> None:
        for klass, nom in cls._handlers.get(event, []):
            instance = klass()
            getattr(instance, nom)(*args)

class MonService(EventBus):
    @on("user.created")
    def bienvenue(self, user: str) -> None:
        print(f"> bienvenue {user}")

class Audit(EventBus):
    @on("user.created")
    def log(self, user: str) -> None:
        print(f"> audit: création de {user}")

EventBus.publish("user.created", "Ada")
```

</details>


---

## Ressources

- [PEP 487 — `__set_name__` et `__init_subclass__`](https://peps.python.org/pep-0487/)
- Brett Cannon — *Why you need `__init_subclass__`*
- Raymond Hettinger — *Descriptor HowTo* (mentionne l'interaction)
